In [1]:
import os

from google.colab import userdata
token = userdata.get("KAGGLE_API_TOKEN")

!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

In [2]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
!git clone https://{TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 31 (delta 11), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 183.79 KiB | 9.19 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/walmart-forecasting


In [3]:
!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
import os

data_dir = "data"

while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

os.listdir(data_dir)

100% 2.70M/2.70M [00:00<00:00, 153MB/s]



['sampleSubmission.csv', 'stores.csv', 'features.csv', 'train.csv', 'test.csv']

In [5]:
# %pip install -q dagshub
# !pip install mlflow --quiet

import dagshub
import mlflow

dagshub.init(repo_owner='amama22', repo_name='walmart-forecasting', mlflow=True)

mlflow.set_experiment("LightGBM_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=369a02bc-7f97-42d7-a085-386395b06a88&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=86e34f85ea8e87d99c9663cc6aea76637338568953a2387aec1b2afeb7fe41e7




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/6f7e968e8d7a4136873ea7714d22a591', creation_time=1783749229644, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783749229644, lifecycle_stage='active', name='LightGBM_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [6]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
import mlflow

from src.data_prep import load_raw_data, merge_all, clean_data
from src.feature_engineering import add_lag_features, add_holiday_proximity, add_expanding_dept_avg, add_expanding_store_avg
from src.evaluation import wmae, walk_forward_splits

train, test, features, stores = load_raw_data(data_dir="data")

train_merged = merge_all(train, features, stores)
test_merged = merge_all(test, features, stores)

print("train_merged shape:", train_merged.shape)
print("test_merged shape:", test_merged.shape)
train_merged.head()

train_merged shape: (421570, 16)
test_merged shape: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [6]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):

    null_counts_before = train_merged.isnull().sum()
    null_counts_before = null_counts_before[null_counts_before > 0]
    print("Nulls before cleaning:\n", null_counts_before)

    mlflow.log_param("markdown_fill_strategy", "fillna_0")
    mlflow.log_param("cpi_unemployment_fill_strategy", "ffill_per_store")

    for col, n in null_counts_before.items():
        mlflow.log_metric(f"nulls_before_{col}", int(n))

    train_merged = clean_data(train_merged)
    test_merged = clean_data(test_merged)

    null_counts_after = train_merged.isnull().sum()
    null_counts_after = null_counts_after[null_counts_after > 0]
    print("\nNulls after cleaning:\n", null_counts_after)

    for col in null_counts_before.index:
        mlflow.log_metric(f"nulls_after_{col}", int(train_merged[col].isnull().sum()))

print("train_merged shape:", train_merged.shape)
print("test_merged shape:", test_merged.shape)

Nulls before cleaning:
 MarkDown1    270889
MarkDown2    310322
MarkDown3    284479
MarkDown4    286603
MarkDown5    270138
dtype: int64

Nulls after cleaning:
 Series([], dtype: int64)
🏃 View run LightGBM_Cleaning at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/8f6c62d828d44ad6a2426da998816b70
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0
train_merged shape: (421570, 16)
test_merged shape: (115064, 15)


In [7]:
train_fe = train_merged.copy()

train_fe["Year"] = train_fe["Date"].dt.year
train_fe["Month"] = train_fe["Date"].dt.month
train_fe["WeekOfYear"] = train_fe["Date"].dt.isocalendar().week.astype(int)

prior_year = train_fe[["Store", "Dept", "Year", "WeekOfYear", "Weekly_Sales"]].copy()
prior_year["Year"] = prior_year["Year"] + 1
prior_year = prior_year.rename(columns={"Weekly_Sales": "sales_lag_52wk"})

train_fe = train_fe.merge(
    prior_year[["Store", "Dept", "Year", "WeekOfYear", "sales_lag_52wk"]],
    on=["Store", "Dept", "Year", "WeekOfYear"],
    how="left",
)

super_bowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])

train_fe["IsSuperBowl"] = train_fe["Date"].isin(super_bowl)
train_fe["IsLaborDay"] = train_fe["Date"].isin(labor_day)
train_fe["IsThanksgiving"] = train_fe["Date"].isin(thanksgiving)
train_fe["IsChristmas"] = train_fe["Date"].isin(christmas)

train_fe = add_holiday_proximity(train_fe)
train_fe = add_expanding_dept_avg(train_fe)
train_fe = add_expanding_store_avg(train_fe)

for col in ["Store", "Dept", "Type"]:
    train_fe[col] = train_fe[col].astype("category")

print("train_fe shape:", train_fe.shape)
print("sales_lag_52wk nulls:", train_fe["sales_lag_52wk"].isnull().sum())
print("\nholiday flag counts:")
print(train_fe[["IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas"]].sum())
train_fe.head()

train_fe shape: (421570, 27)
sales_lag_52wk nulls: 160029

holiday flag counts:
IsSuperBowl       8895
IsLaborDay        8861
IsThanksgiving    5959
IsChristmas       5946
dtype: int64


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Month,WeekOfYear,sales_lag_52wk,IsSuperBowl,IsLaborDay,IsThanksgiving,IsChristmas,days_to_nearest_holiday,dept_avg_sales_expanding,store_avg_sales_expanding
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,...,2,5,NaN,False,False,False,False,7,NaN,NaN
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,...,2,6,NaN,True,False,False,False,0,19596.298000,22516.313699
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,...,2,7,NaN,False,False,False,False,-7,25989.064556,22660.639072
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,...,2,8,NaN,False,False,False,False,-14,25609.430889,22467.677965
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,...,3,9,NaN,False,False,False,False,-21,22992.581944,21745.645939


In [8]:
FEATURE_COLS = [
    "Store", "Dept", "Type", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "Year", "Month", "WeekOfYear",
    "sales_lag_52wk",
    "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
    "days_to_nearest_holiday",
    "dept_avg_sales_expanding", "store_avg_sales_expanding",
]
TARGET = "Weekly_Sales"

splits = walk_forward_splits(train_fe, n_splits=3, horizon_weeks=38)
print(f"Number of CV folds: {len(splits)}")

with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    mlflow.log_param("n_splits", len(splits))
    mlflow.log_param("n_features", len(FEATURE_COLS))

    fold_wmaes = []
    importances = np.zeros(len(FEATURE_COLS))

    for i, (train_mask, val_mask) in enumerate(splits):
        X_train, y_train = train_fe.loc[train_mask, FEATURE_COLS], train_fe.loc[train_mask, TARGET]
        X_val, y_val = train_fe.loc[val_mask, FEATURE_COLS], train_fe.loc[val_mask, TARGET]

        model = LGBMRegressor(random_state=42, verbose=-1)
        model.fit(X_train, y_train, categorical_feature=["Store", "Dept", "Type"])

        preds = model.predict(X_val)
        fold_wmae = wmae(y_val, preds, train_fe.loc[val_mask, "IsHoliday"])
        fold_wmaes.append(fold_wmae)
        importances += model.feature_importances_

        mlflow.log_metric(f"fold{i}_wmae", fold_wmae)
        print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    mean_wmae = np.mean(fold_wmaes)
    mlflow.log_metric("mean_wmae", mean_wmae)
    print(f"\nMean WMAE (all features): {mean_wmae:.2f}")

    importance_df = pd.DataFrame({"feature": FEATURE_COLS, "importance": importances}).sort_values("importance", ascending=False)
    print("\nFeature importances (summed across folds):")
    print(importance_df.to_string(index=False))

    importance_df.to_csv("feature_importance_lightgbm.csv", index=False)
    mlflow.log_artifact("feature_importance_lightgbm.csv")

Number of CV folds: 3
Fold 0: WMAE = 4108.30
Fold 1: WMAE = 2739.35
Fold 2: WMAE = 1974.49

Mean WMAE (all features): 2940.71

Feature importances (summed across folds):
                  feature  importance
                     Dept      3055.0
                    Store      2411.0
 dept_avg_sales_expanding       846.0
           sales_lag_52wk       542.0
               WeekOfYear       465.0
store_avg_sales_expanding       335.0
  days_to_nearest_holiday       205.0
                     Type       204.0
                     Size       165.0
              Temperature       141.0
                      CPI       120.0
               Fuel_Price       111.0
                    Month       107.0
           IsThanksgiving        92.0
             Unemployment        81.0
                MarkDown3        40.0
                     Year        30.0
                MarkDown1        20.0
                MarkDown5        14.0
                MarkDown4        10.0
                MarkDown2       

In [9]:
PRUNED_FEATURE_COLS = [c for c in FEATURE_COLS if c not in
    ["IsChristmas", "IsLaborDay", "IsSuperBowl", "MarkDown1", "MarkDown2", "MarkDown4", "MarkDown5", "Year"]]
print(f"Pruned from {len(FEATURE_COLS)} to {len(PRUNED_FEATURE_COLS)} features")

with mlflow.start_run(run_name="LightGBM_Feature_Selection_Pruned"):
    mlflow.log_param("n_splits", len(splits))
    mlflow.log_param("n_features", len(PRUNED_FEATURE_COLS))
    mlflow.log_param("dropped_features", ",".join(sorted(set(FEATURE_COLS) - set(PRUNED_FEATURE_COLS))))

    fold_wmaes_pruned = []
    for i, (train_mask, val_mask) in enumerate(splits):
        X_train, y_train = train_fe.loc[train_mask, PRUNED_FEATURE_COLS], train_fe.loc[train_mask, TARGET]
        X_val, y_val = train_fe.loc[val_mask, PRUNED_FEATURE_COLS], train_fe.loc[val_mask, TARGET]

        model = LGBMRegressor(random_state=42, verbose=-1)
        model.fit(X_train, y_train, categorical_feature=["Store", "Dept", "Type"])

        preds = model.predict(X_val)
        fold_wmae = wmae(y_val, preds, train_fe.loc[val_mask, "IsHoliday"])
        fold_wmaes_pruned.append(fold_wmae)
        mlflow.log_metric(f"fold{i}_wmae", fold_wmae)
        print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    mean_wmae_pruned = np.mean(fold_wmaes_pruned)
    mlflow.log_metric("mean_wmae", mean_wmae_pruned)
    print(f"\nMean WMAE (pruned features): {mean_wmae_pruned:.2f}")
    print(f"Mean WMAE (all features):    {mean_wmae:.2f}")

Pruned from 24 to 16 features
Fold 0: WMAE = 4108.30
Fold 1: WMAE = 2680.90
Fold 2: WMAE = 1953.66

Mean WMAE (pruned features): 2914.29
Mean WMAE (all features):    2940.71
🏃 View run LightGBM_Feature_Selection_Pruned at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/caf0d01c68704badaea2113d3051f8bc
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


In [10]:
%pip install -q optuna
import optuna

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "random_state": 42,
        "verbose": -1,
    }

    fold_wmaes = []
    for train_mask, val_mask in splits:
        X_train, y_train = train_fe.loc[train_mask, PRUNED_FEATURE_COLS], train_fe.loc[train_mask, TARGET]
        X_val, y_val = train_fe.loc[val_mask, PRUNED_FEATURE_COLS], train_fe.loc[val_mask, TARGET]

        model = LGBMRegressor(**params)
        model.fit(X_train, y_train, categorical_feature=["Store", "Dept", "Type"])
        preds = model.predict(X_val)
        fold_wmaes.append(wmae(y_val, preds, train_fe.loc[val_mask, "IsHoliday"]))

    mean_wmae = np.mean(fold_wmaes)

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_wmae", mean_wmae)

    return mean_wmae

with mlflow.start_run(run_name="LightGBM_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_mean_wmae", study.best_value)
    print("Best WMAE:", study.best_value)
    print("Best params:", study.best_params)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.6 MB/s eta 0:00:00


[I 2026-07-11 12:45:46,408] A new study created in memory with name: no-name-8e8a2ced-dcfb-4af0-8db9-b436e923f9df
[I 2026-07-11 12:46:43,733] Trial 0 finished with value: 2589.0603097055205 and parameters: {'num_leaves': 51, 'learning_rate': 0.03659335390202527, 'n_estimators': 536, 'min_child_samples': 34, 'feature_fraction': 0.7699718516559515, 'bagging_fraction': 0.7625240848617834, 'bagging_freq': 1, 'lambda_l1': 1.7714813741743275e-07, 'lambda_l2': 1.1079085300035188e-08}. Best is trial 0 with value: 2589.0603097055205.


🏃 View run trial_0 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/8195e15ffa7142498d55010461794f7e
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


[I 2026-07-11 12:48:39,012] Trial 1 finished with value: 2715.081360225246 and parameters: {'num_leaves': 207, 'learning_rate': 0.19159283447883035, 'n_estimators': 543, 'min_child_samples': 68, 'feature_fraction': 0.7238583117117575, 'bagging_fraction': 0.6770530806153615, 'bagging_freq': 5, 'lambda_l1': 1.785653689585821e-07, 'lambda_l2': 0.00621895371006522}. Best is trial 0 with value: 2589.0603097055205.


🏃 View run trial_1 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/5bebe97bf8584048a6b25eaf619d8573
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


...

🏃 View run trial_27 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/35df47ece937484bb989d43d3c0d28fd
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


[I 2026-07-11 13:35:16,198] Trial 28 finished with value: 2517.9742716865576 and parameters: {'num_leaves': 233, 'learning_rate': 0.020828964935767205, 'n_estimators': 731, 'min_child_samples': 39, 'feature_fraction': 0.6282701081840262, 'bagging_fraction': 0.9680075913381742, 'bagging_freq': 7, 'lambda_l1': 1.1337863237293077e-08, 'lambda_l2': 3.358719631913887e-07}. Best is trial 20 with value: 2502.852451709473.


🏃 View run trial_28 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/5ad824d2c65a439ba196131a101adc48
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


[I 2026-07-11 13:37:29,172] Trial 29 finished with value: 2477.74200901786 and parameters: {'num_leaves': 187, 'learning_rate': 0.03637469659448251, 'n_estimators': 659, 'min_child_samples': 33, 'feature_fraction': 0.6624055100034115, 'bagging_fraction': 0.917448839889876, 'bagging_freq': 6, 'lambda_l1': 8.789112668083663e-05, 'lambda_l2': 1.0324546326772466e-08}. Best is trial 29 with value: 2477.74200901786.


🏃 View run trial_29 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/0487e77af8744c20a692cfcdd6290934
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0
Best WMAE: 2477.74200901786
Best params: {'num_leaves': 187, 'learning_rate': 0.03637469659448251, 'n_estimators': 659, 'min_child_samples': 33, 'feature_fraction': 0.6624055100034115, 'bagging_fraction': 0.917448839889876, 'bagging_freq': 6, 'lambda_l1': 8.789112668083663e-05, 'lambda_l2': 1.0324546326772466e-08}
🏃 View run LightGBM_HPO at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/f77629fad1624f84a33803d01a79f59d
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

class WalmartFeatureBuilder(BaseEstimator, TransformerMixin):

    def __init__(self, features_df, stores_df):
        self.features_df = features_df
        self.stores_df = stores_df

    def fit(self, X, y=None):
            df = X.copy()
            df["Date"] = pd.to_datetime(df["Date"])
            df["Year"] = df["Date"].dt.year
            df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

            if y is not None:
                df["Weekly_Sales"] = np.asarray(y)

            self.history_ = df[["Store", "Dept", "Year", "WeekOfYear", "Weekly_Sales"]].copy()

            # exclude each group's final week before averaging, to exactly match
            # the shift(1)-before-expanding-mean semantics used during training
            dept_weekly = df.groupby(["Dept", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Dept", "Date"])
            self.dept_avg_final_ = (
                dept_weekly.groupby("Dept")
                .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())
            ).to_dict()

            store_weekly = df.groupby(["Store", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Store", "Date"])
            self.store_avg_final_ = (
                store_weekly.groupby("Store")
                .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())
            ).to_dict()

            return self

    def transform(self, X):
        df = merge_all(X, self.features_df, self.stores_df)
        df = clean_data(df)

        df["Date"] = pd.to_datetime(df["Date"])
        df["Month"] = df["Date"].dt.month
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
        df["Year"] = df["Date"].dt.year

        thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
        df["IsThanksgiving"] = df["Date"].isin(thanksgiving)

        lookup = self.history_.copy()
        lookup["Year"] = lookup["Year"] + 1
        lookup = lookup.rename(columns={"Weekly_Sales": "sales_lag_52wk"})
        df = df.merge(lookup, on=["Store", "Dept", "Year", "WeekOfYear"], how="left")

        df = add_holiday_proximity(df)

        df["dept_avg_sales_expanding"] = df["Dept"].map(self.dept_avg_final_)
        df["store_avg_sales_expanding"] = df["Store"].map(self.store_avg_final_)

        for col in ["Store", "Dept", "Type"]:
            df[col] = df[col].astype("category")

        return df[PRUNED_FEATURE_COLS]

In [14]:
from sklearn.pipeline import Pipeline
train_mask, val_mask = splits[-1]  # most recent fold, closest analog to real test

X_train_raw = train.loc[train_mask, ["Store", "Dept", "Date", "IsHoliday"]]
y_train_raw = train.loc[train_mask, "Weekly_Sales"]

X_val_raw = train.loc[val_mask, ["Store", "Dept", "Date", "IsHoliday"]]
y_val_raw = train.loc[val_mask, "Weekly_Sales"]

best_params = study.best_params

check_pipeline = Pipeline([
    ("features", WalmartFeatureBuilder(features, stores)),
    ("model", LGBMRegressor(**best_params, random_state=42, verbose=-1)),
])

check_pipeline.fit(X_train_raw, y_train_raw, model__categorical_feature=["Store", "Dept", "Type"])
val_preds = check_pipeline.predict(X_val_raw)

check_wmae = wmae(y_val_raw, val_preds, X_val_raw["IsHoliday"])
print(f"Pipeline WMAE on held-out fold: {check_wmae:.2f}")
print(f"(for reference, flat-DataFrame tuned CV mean was: {study.best_value:.2f})")

/tmp/ipykernel_858/3870134699.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())
/tmp/ipykernel_858/3870134699.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())


Pipeline WMAE on held-out fold: 1665.13
(for reference, flat-DataFrame tuned CV mean was: 2477.74)


In [15]:
X_full = train[["Store", "Dept", "Date", "IsHoliday"]]
y_full = train["Weekly_Sales"]

final_pipeline = Pipeline([
    ("features", WalmartFeatureBuilder(features, stores)),
    ("model", LGBMRegressor(**best_params, random_state=42, verbose=-1)),
])

with mlflow.start_run(run_name="LightGBM_Final_Pipeline"):
    final_pipeline.fit(X_full, y_full, model__categorical_feature=["Store", "Dept", "Type"])

    mlflow.log_params(best_params)
    mlflow.log_param("n_features", len(PRUNED_FEATURE_COLS))
    mlflow.log_param("features", ",".join(PRUNED_FEATURE_COLS))

    # reference metric from the CV/HPO stage -- this pipeline hasn't been
    # re-validated on a held-out fold itself (it's fit on ALL of train),
    # so this is the honest generalization estimate to report, not an
    # in-sample number computed against data the model has already seen
    mlflow.log_metric("cv_mean_wmae_reference", study.best_value)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="lightgbm_pipeline",
        registered_model_name="walmart-lightgbm-store-sales",
    )

    print("Logged and registered as 'walmart-lightgbm-store-sales'")
    print(f"Reference CV WMAE: {study.best_value:.2f}")

/tmp/ipykernel_858/3870134699.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())
/tmp/ipykernel_858/3870134699.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean())
2026/07/11 14:43:37 WARNING mlflow.models.model: `

🏃 View run LightGBM_Final_Pipeline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/c0ba8dc948004180b92ea048d44abc31
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


MlflowException: The sklearn model could not be serialized in the skops serialization format. skops does not support custom functions or classes that are not defined at the top level. To work around this limitation, you can set the serialization_format 'cloudpickle', while exercising caution due to the possible arbitrary code during model deserialization using CloudPickle.

In [16]:
with mlflow.start_run(run_name="LightGBM_Final_Pipeline"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_features", len(PRUNED_FEATURE_COLS))
    mlflow.log_param("features", ",".join(PRUNED_FEATURE_COLS))
    mlflow.log_metric("cv_mean_wmae_reference", study.best_value)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="lightgbm_pipeline",
        registered_model_name="walmart-lightgbm-store-sales",
        serialization_format="cloudpickle",
    )

    print("Logged and registered as 'walmart-lightgbm-store-sales'")
    print(f"Reference CV WMAE: {study.best_value:.2f}")

2026/07/11 14:49:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 14:49:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'walmart-lightgbm-store-sales'.
2026/07/11 14:49:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-lightgbm-store-sales, version 1
Created version '1' of model 'walmart-lightgbm-store-sales'.


Logged and registered as 'walmart-lightgbm-store-sales'
Reference CV WMAE: 2477.74
🏃 View run LightGBM_Final_Pipeline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0/runs/71e2f38eabef45fc8188e436bf50fc44
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/0


In [17]:
test_raw = test[["Store", "Dept", "Date", "IsHoliday"]]

test_preds = final_pipeline.predict(test_raw)
test_preds = np.clip(test_preds, 0, None)  # Weekly_Sales can't be negative

submission = pd.DataFrame({
    "Id": test["Store"].astype(str) + "_" + test["Dept"].astype(str) + "_" + test["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_preds,
})

submission.to_csv("submission_lightgbm.csv", index=False)
print(submission.shape)
submission.head()

(115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,40139.952038
1,1_1_2012-11-09,20029.787120
2,1_1_2012-11-16,20190.018770
3,1_1_2012-11-23,20387.926239
4,1_1_2012-11-30,24681.899574


In [18]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_lightgbm.csv -m "LightGBM baseline, tuned, WMAE~2477 CV"

100% 3.80M/3.80M [00:00<00:00, 4.82MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting